# Example 03: Regional non-Hermitian retarded top bath (pedagogical minimal notebook)

This notebook keeps the **experimental regional bath path** and focuses on skin-effect diagnostics from
\[
H_{\mathrm{eff}} = H_{\mathrm{device}} + \Sigma^R_{\mathrm{top}}.
\]
We compare two cases only: **A:** `gammay = 0` and **B:** `gammay \neq 0`.


## 1) Introduction
Primary goal: identify boundary accumulation trends from the effective non-Hermitian spectrum and eigenmodes, then compare with compact lead-transport observables.


## Imports


In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__))))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics
using PyPlot


## 2) User-editable parameters
Edit `Nx`, `Ny`, `gamma0`, and `gammay_caseB` directly here. Small lattices are fast; quasi-1D (`Ny=1,2` and larger `Nx`) is useful for wire-like comparisons.


In [ ]:
quick_test = true
run_optional_timecheck = true

# Geometry (keep these directly editable).
if quick_test
    Nx, Ny = 6, 2
    tmax, dt = 18.0, 1.0
    sweep_points = 5
else
    Nx, Ny = 10, 2
    tmax, dt = 32.0, 0.5
    sweep_points = 9
end

# model
Nσ = 2
N_orb = 1
γ = 1.0
γso = 0.50 + 0.0im
μ = 0.0
Bz = 0.15

# lead / poles
β = 33.0
N_λ1 = 49
N_λ2 = 20

# regional retarded bath
gamma0 = 0.20
gammay_caseA = 0.0
gammay_caseB = 0.10

# solver controls
reltol = 1e-6
abstol = 1e-8

# lightweight sweep parameter (conductance-like proxy section)
Bz_sweep = collect(range(0.00, 0.30; length=sweep_points))

# restrained academic palette
PALETTE = Dict(
    :navy => "#1f3b73",
    :teal => "#2a9d8f",
    :cyan => "#5bc0be",
    :gold => "#e9c46a",
    :orange => "#f4a261",
    :lightbg => "#f7f8fa",
)

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = PALETTE[:lightbg]
plt.rcParams["axes.grid"] = true
plt.rcParams["grid.alpha"] = 0.25

println("Nx=$(Nx), Ny=$(Ny), quick_test=$(quick_test)")
println("Case A gammay=$(gammay_caseA), Case B gammay=$(gammay_caseB)")
println("Check gamma0 >= |gammay_caseB| : ", gamma0 >= abs(gammay_caseB))


## 3) Geometry and bath-coupled sites
The added retarded bath is coupled **only to the top row** (`y = Ny`) for all `x`. We visualize top, bottom, and bath-coupled sites on the lattice so boundary diagnostics are unambiguous.


In [ ]:
@inline local_index(x::Int, y::Int, Ny::Int) = (x - 1) * Ny + y

@inline function local_subrange(site::Int, N_loc::Int)
    a = (site - 1) * N_loc + 1
    b = site * N_loc
    return a:b
end

function top_edge_coupled_indices(; Nx::Int, Ny::Int, N_loc::Int)
    top_sites = [local_index(x, Ny, Ny) for x in 1:Nx]
    idx_couple = Int[]
    for s in top_sites
        append!(idx_couple, collect(local_subrange(s, N_loc)))
    end
    return top_sites, idx_couple
end

function bottom_edge_indices(; Nx::Int, Ny::Int)
    [local_index(x, 1, Ny) for x in 1:Nx]
end

function site_scalar_to_grid(values::AbstractVector, Nx::Int, Ny::Int)
    @assert length(values) == Nx * Ny
    [values[local_index(x, y, Ny)] for y in Ny:-1:1, x in 1:Nx]
end


### Geometry map: top vs bottom vs bath-coupled sites
Why this plot: it confirms exactly where the non-Hermitian bath acts. Expected effect: mode weights sensitive to bath-induced non-Hermiticity should preferentially accumulate/deplete near the top boundary.


In [ ]:
N_sites = Nx * Ny
N_loc = Nσ * N_orb

top_sites_geom, idx_couple_geom = top_edge_coupled_indices(; Nx, Ny, N_loc)
bottom_sites_geom = bottom_edge_indices(; Nx, Ny)

site_class = zeros(Float64, N_sites)
site_class[top_sites_geom] .= 1.0
site_class[bottom_sites_geom] .= -1.0

fig, ax = plt.subplots(figsize=(6.2, 3.2))
img = ax.imshow(site_scalar_to_grid(site_class, Nx, Ny), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_title(raw"$\mathrm{Geometry:\ top\ row\ (bath),\ bottom\ row\ (reference)}$")
ax.set_xlabel(raw"$x$")
ax.set_ylabel(raw"$y\,(\mathrm{top\ at\ top})$")
plt.colorbar(img, ax=ax, label=raw"$+1\!:\mathrm{top},\ -1\!:\mathrm{bottom}$")
plt.show()

println("top sites       = ", top_sites_geom)
println("bottom sites    = ", bottom_sites_geom)
println("idx_couple size = ", length(idx_couple_geom), " (DOF indices)")


### Build model helpers and two cases
We keep the architecture unchanged: the added bath remains in `ExtraBathSpec` under `ExperimentalBlockRHSParams` (retarded-only extension).


In [ ]:
function add_uniform_onsite!(H::AbstractMatrix{ComplexF64}; Nx::Int, Ny::Int, N_loc::Int, μ::Float64, Bz::Float64)
    σz = ComplexF64[1 0; 0 -1]
    onsite = (-μ) * Matrix{ComplexF64}(I, N_loc, N_loc) + Bz * σz
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        H[r, r] .+= onsite
    end
    return H
end

function build_two_terminal_blocks(p_model::ModelParamsTDNEGF; β::Float64, N_λ1::Int, N_λ2::Int)
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)
    Σᴸ_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    Σᴳ_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=p_model.Nx, y_coup=1:p_model.Ny)
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=1, y_coup=1:p_model.Ny)
    left_block = SelfEnergyBlock(:left, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anR)
    return [left_block, right_block]
end

function build_top_bath_ΣR(; n_coupled_sites::Int, N_loc::Int, gamma0::Float64, gammay::Float64)
    @assert N_loc == 2
    @assert gamma0 >= 0
    if gamma0 < abs(gammay)
        @warn "gamma0 < |gammay|: possible effective gain channel."
    end
    σy = ComplexF64[0 -im; im 0]
    Σ_site = -1im .* (gamma0 .* Matrix{ComplexF64}(I, N_loc, N_loc) .+ gammay .* σy)
    ΣR_loc = kron(Matrix{ComplexF64}(I, n_coupled_sites, n_coupled_sites), Σ_site)
    return Σ_site, ΣR_loc
end

function embed_local_block(idx_couple::Vector{Int}, block_local::AbstractMatrix{ComplexF64}, Nglobal::Int)
    Σ_global = zeros(ComplexF64, Nglobal, Nglobal)
    Σ_global[idx_couple, idx_couple] .= block_local
    return Σ_global
end

function make_case(; Nx::Int, Ny::Int, Nσ::Int, N_orb::Int, γ::Float64, γso, β::Float64, N_λ1::Int, N_λ2::Int,
                   μ::Float64, Bz::Float64, gamma0::Float64, gammay::Float64)
    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb, Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    H_device = build_H_ab(; Nx, Ny, Nσ, N_orb, γ=γ, γso=γso)
    add_uniform_onsite!(H_device; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc, μ=μ, Bz=Bz)
    p_model.H_ab .= H_device
    p_model.H0_ab .= H_device

    blocks = build_two_terminal_blocks(p_model; β=β, N_λ1=N_λ1, N_λ2=N_λ2)
    Δ_blocks = ComplexF64[0.5 + 0.0im, -0.5 + 0.0im]

    top_sites, idx_couple = top_edge_coupled_indices(; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc)
    Σ_site, ΣR_loc = build_top_bath_ΣR(; n_coupled_sites=length(top_sites), N_loc=p_model.N_loc, gamma0=gamma0, gammay=gammay)
    ΣR_top_global = embed_local_block(idx_couple, ΣR_loc, size(H_device, 1))

    top_bath = TDNEGF.ExtraBathSpec(:nonhermitian_retarded, :top_edge_nonhermitian, idx_couple, ΣR_loc, true)
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model, [top_bath])
    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    return (; p_model, p_blocks, u0, top_sites, idx_couple, Σ_site, ΣR_loc, ΣR_top_global, gammay, Bz)
end

caseA = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseA)
caseB = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseB)

println("Case A/B ready.")


### Sanity checks for coupled region and ΣR embedding
Why this printout: verifies that the local retarded bath block and global embedding dimensions are consistent with the intended coupled sites.


In [ ]:
println("gamma0 >= |gammay_caseA| : ", gamma0 >= abs(gammay_caseA))
println("gamma0 >= |gammay_caseB| : ", gamma0 >= abs(gammay_caseB))
println("top-edge sites         : ", caseA.top_sites)
println("idx_couple length      : ", length(caseA.idx_couple))
println("local ΣR shape         : ", size(caseA.ΣR_loc))
println("embedded/global ΣR     : ", size(caseA.ΣR_top_global))
println("affected sites (A,B)   : ", caseA.top_sites, " ; ", caseB.top_sites)


## 4) Effective non-Hermitian Hamiltonian
We now diagnose how the added retarded bath modifies the finite-device spectrum. Complex eigenvalues reveal damping/amplification trends through \(\mathrm{Im}(E)\).


In [ ]:
heff(case_setup) = Matrix(case_setup.p_model.H_ab) + case_setup.ΣR_top_global

valsA = eigen(heff(caseA)).values
valsB = eigen(heff(caseB)).values


### Complex spectrum: \(\mathrm{Re}(E)\) vs \(\mathrm{Im}(E)\)
Why this plot: it is the direct non-Hermitian fingerprint of the regional bath. Expected effect: turning on `gammay` shifts and redistributes imaginary parts and can split/tilt spectral structures associated with spin-selective dissipation.


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.scatter(real.(valsA), imag.(valsA), s=26, color=PALETTE[:navy], alpha=0.85, label=raw"$\mathrm{Case\ A}:\ \gamma_y=0$")
ax.scatter(real.(valsB), imag.(valsB), s=26, color=PALETTE[:orange], alpha=0.75, label=raw"$\mathrm{Case\ B}:\ \gamma_y\neq 0$")
ax.axhline(0.0, color=PALETTE[:cyan], lw=1.0)
ax.set_xlabel(raw"$\mathrm{Re}(E)$")
ax.set_ylabel(raw"$\mathrm{Im}(E)$")
ax.set_title(raw"$\mathrm{Complex\ spectrum\ of}\ H_{\mathrm{eff}}$")
ax.legend(frameon=false, fontsize=9)
plt.show()


## 5) Selected eigenmodes and spatial accumulation
**Mode-selection rule (explicit and reproducible):** choose the modes with the largest imaginary parts, i.e. `arg sort(Im(E))` from the top. This targets least-damped / most bath-sensitive right eigenmodes.


In [ ]:
function site_weights_from_state(vec_state::AbstractVector, Nx::Int, Ny::Int, N_loc::Int)
    weights = zeros(Float64, Nx * Ny)
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        weights[site] = real(sum(abs2, vec_state[r]))
    end
    s = sum(weights)
    return s > 0 ? weights ./ s : weights
end

function select_modes_largest_im(vals; n_modes::Int=3)
    ord = sortperm(imag.(vals))
    return reverse(ord[max(1, end-n_modes+1):end])
end

function mode_maps(case_setup; n_modes::Int=3)
    ev = eigen(heff(case_setup))
    vals, vecs = ev.values, ev.vectors
    sel = select_modes_largest_im(vals; n_modes=n_modes)
    maps = [site_scalar_to_grid(site_weights_from_state(vecs[:,i], case_setup.p_model.Nx, case_setup.p_model.Ny, case_setup.p_model.N_loc),
                                case_setup.p_model.Nx, case_setup.p_model.Ny) for i in sel]
    return vals, sel, maps
end

valsA_full, selA, mapsA = mode_maps(caseA; n_modes=3)
valsB_full, selB, mapsB = mode_maps(caseB; n_modes=3)

println("Selected modes (A): ", selA, " with Im(E)=", imag.(valsA_full[selA]))
println("Selected modes (B): ", selB, " with Im(E)=", imag.(valsB_full[selB]))


### Lattice-resolved right-eigenmode weights (Case A)
Why this plot: spatial localization of right modes is a direct skin-effect-style indicator. Expected effect: with scalar damping (`gammay=0`), accumulation should mainly reflect geometry and uniform loss structure.


In [ ]:
fig, axs = plt.subplots(1, length(mapsA), figsize=(4.2*length(mapsA), 3.4), constrained_layout=true)
if length(mapsA) == 1
    axs = [axs]
end
for j in 1:length(mapsA)
    img = axs[j].imshow(mapsA[j], cmap="cividis", aspect="auto")
    axs[j].set_title("Case A mode $(selA[j]), Im(E)=$(round(imag(valsA_full[selA[j]]), digits=3))")
    axs[j].set_xlabel(raw"$x$")
    axs[j].set_ylabel(raw"$y$")
    plt.colorbar(img, ax=axs[j], fraction=0.046)
end
plt.show()


### Lattice-resolved right-eigenmode weights (Case B)
Why this plot: compare against Case A with the **same selection rule**. Expected effect: turning on `gammay` can change where high-`Im(E)` modes accumulate and can enhance directional/boundary imbalance.


In [ ]:
fig, axs = plt.subplots(1, length(mapsB), figsize=(4.2*length(mapsB), 3.4), constrained_layout=true)
if length(mapsB) == 1
    axs = [axs]
end
for j in 1:length(mapsB)
    img = axs[j].imshow(mapsB[j], cmap="cividis", aspect="auto")
    axs[j].set_title("Case B mode $(selB[j]), Im(E)=$(round(imag(valsB_full[selB[j]]), digits=3))")
    axs[j].set_xlabel(raw"$x$")
    axs[j].set_ylabel(raw"$y$")
    plt.colorbar(img, ax=axs[j], fraction=0.046)
end
plt.show()


### Build observables for transport comparisons (single run per case)
We now run one lightweight time evolution per case and reuse these observables for charge/spin transport and optional checks.


In [ ]:
function run_case(case_setup; tspan=(0.0, 20.0), reltol=1e-6, abstol=1e-8, dt_save=1.0)
    p_model, p_blocks, u0 = case_setup.p_model, case_setup.p_blocks, case_setup.u0
    save_times = collect(range(tspan[1], tspan[2]; step=dt_save))
    prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)
    sol = solve(prob, Vern7(); adaptive=true, saveat=save_times, dense=false, reltol=reltol, abstol=abstol)

    obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
    obs.t = sol.t
    for (it, ut) in enumerate(sol.u)
        obs.idx = it
        ptr = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
        obs_n_i!(ptr, p_model, obs)
        obs_σ_i!(ptr, p_model, obs)
        obs_Ixα!(ptr, p_blocks, obs)
    end
    return sol, obs
end

solA, obsA = run_case(caseA; tspan=(0.0, tmax), reltol=reltol, abstol=abstol, dt_save=dt)
solB, obsB = run_case(caseB; tspan=(0.0, tmax), reltol=reltol, abstol=abstol, dt_save=dt)

println("dynamics done: Nt(A)=$(length(obsA.t)), Nt(B)=$(length(obsB.t))")


## 6) Lead charge currents
Why this plot: lead charge currents provide the transport-side counterpart of non-Hermitian mode reshaping. Expected effect: `gammay` can modify left/right current imbalance and transient approach to steady behavior.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.8, 4.0), sharey=true, constrained_layout=true)

axs[1].plot(obsA.t, obsA.Iα[1, :], color=PALETTE[:navy], lw=2.0, label=raw"$I_{L}$")
axs[1].plot(obsA.t, obsA.Iα[2, :], color=PALETTE[:teal], lw=2.0, ls="--", label=raw"$I_{R}$")
axs[1].set_title(raw"$\mathrm{Case\ A}:\ \gamma_y=0$")
axs[1].set_xlabel(raw"$t$")
axs[1].set_ylabel(raw"$I_{\alpha}\;(\mathrm{arb.})$")
axs[1].legend(frameon=false, fontsize=9)

axs[2].plot(obsB.t, obsB.Iα[1, :], color=PALETTE[:orange], lw=2.0, label=raw"$I_{L}$")
axs[2].plot(obsB.t, obsB.Iα[2, :], color=PALETTE[:cyan], lw=2.0, ls="--", label=raw"$I_{R}$")
axs[2].set_title(raw"$\mathrm{Case\ B}:\ \gamma_y\neq 0$")
axs[2].set_xlabel(raw"$t$")
axs[2].legend(frameon=false, fontsize=9)

plt.show()


## 7) Lead spin currents
Why this plot: spin-current components test whether spin-selective non-Hermitian damping (`gammay`) introduces component-dependent transport asymmetries.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.4, 4.2), sharey=true, constrained_layout=true)
comps = [(1, raw"$I^{x}_{\alpha}$", PALETTE[:navy]), (2, raw"$I^{y}_{\alpha}$", PALETTE[:teal]), (3, raw"$I^{z}_{\alpha}$", PALETTE[:gold])]

for c, lab, col in comps
    axs[1].plot(obsA.t, obsA.Iαx[1, c, :], color=col, lw=1.9, label=lab * raw"$\,(L)$")
    axs[1].plot(obsA.t, obsA.Iαx[2, c, :], color=col, lw=1.9, ls="--", alpha=0.85, label=lab * raw"$\,(R)$")
    axs[2].plot(obsB.t, obsB.Iαx[1, c, :], color=col, lw=1.9, label=lab * raw"$\,(L)$")
    axs[2].plot(obsB.t, obsB.Iαx[2, c, :], color=col, lw=1.9, ls="--", alpha=0.85, label=lab * raw"$\,(R)$")
end

axs[1].set_title(raw"$\mathrm{Case\ A}:\ \gamma_y=0$")
axs[2].set_title(raw"$\mathrm{Case\ B}:\ \gamma_y\neq 0$")
for ax in axs
    ax.set_xlabel(raw"$t$")
    ax.legend(frameon=false, fontsize=7, ncol=2)
end
axs[1].set_ylabel(raw"$I^{s}_{\alpha}\;(\mathrm{arb.})$")
plt.show()


## 8) Conductance-like parameter sweep
We vary **Zeeman field** `Bz` and use a lightweight transport proxy at final time:
\[
G_{\mathrm{proxy}}(B_z) = I_L(t_f)-I_R(t_f).
\]
Why this matters: it tracks left/right transport asymmetry trends that can correlate with non-Hermitian boundary accumulation.


In [ ]:
function final_current_proxy(; Bz_val::Float64, gammay_val::Float64)
    case = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz=Bz_val, gamma0, gammay=gammay_val)
    _, obs = run_case(case; tspan=(0.0, tmax), reltol=reltol, abstol=abstol, dt_save=dt)
    return obs.Iα[1, end] - obs.Iα[2, end]
end

Gproxy_A = [final_current_proxy(; Bz_val=b, gammay_val=gammay_caseA) for b in Bz_sweep]
Gproxy_B = [final_current_proxy(; Bz_val=b, gammay_val=gammay_caseB) for b in Bz_sweep]

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(Bz_sweep, Gproxy_A, marker="o", color=PALETTE[:navy], lw=2.0, label=raw"$\gamma_y=0$")
ax.plot(Bz_sweep, Gproxy_B, marker="s", color=PALETTE[:orange], lw=2.0, label=raw"$\gamma_y\neq 0$")
ax.axhline(0.0, color=PALETTE[:cyan], lw=1.0)
ax.set_xlabel(raw"$B_z$")
ax.set_ylabel(raw"$G_{\mathrm{proxy}} = I_L(t_f)-I_R(t_f)$")
ax.set_title(raw"$\mathrm{Conductance\!	ext{-}\!like\ transport\ proxy\ vs}\ B_z$")
ax.legend(frameon=false)
plt.show()


## 9) Optional short time-evolution check
A compact final-time spin-density comparison on the lattice (one component) is kept as a supporting diagnostic only.


In [ ]:
if run_optional_timecheck
    szA = vec(obsA.σx_i[:, 3, end])
    szB = vec(obsB.σx_i[:, 3, end])
    mapA = site_scalar_to_grid(szA, Nx, Ny)
    mapB = site_scalar_to_grid(szB, Nx, Ny)
    vmax = max(maximum(abs.(mapA)), maximum(abs.(mapB))) + 1e-12

    fig, axs = plt.subplots(1, 2, figsize=(9.2, 3.6), constrained_layout=true)
    imA = axs[1].imshow(mapA, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    imB = axs[2].imshow(mapB, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axs[1].set_title(raw"$\mathrm{Case\ A}:\ s_z(t_f)$")
    axs[2].set_title(raw"$\mathrm{Case\ B}:\ s_z(t_f)$")
    for ax in axs
        ax.set_xlabel(raw"$x$")
        ax.set_ylabel(raw"$y$")
    end
    plt.colorbar(imB, ax=axs, fraction=0.046, label=raw"$s_z$")
    plt.show()
end


## 10) Conclusions
- The top-edge regional retarded bath is explicitly coupled only on `y = Ny`, and geometry checks make this transparent.
- `H_{\mathrm{eff}}` diagnostics (complex spectrum + largest-`Im(E)` right-mode maps) are the main skin-effect indicators in this example.
- Lead charge/spin-current plots and the lightweight `B_z` sweep of `G_{\mathrm{proxy}}` provide transport-side comparison between `gammay=0` and `gammay\neq0`.
- Scope remains intentionally limited to the **retarded-only** regional bath extension.
